In [8]:
import pandas as pd
import numpy as np
from scipy.spatial import KDTree
import os
import time
import h5py
from joblib import Parallel, delayed

In [9]:
def read_hdf(file_path, key):
    return pd.read_hdf(file_path, key=key, encoding='utf-8')


def read_hdf_chunks(file_path, key, chunksize=1_000_000):
    # Read HDF file in chunks and concatenate into a single DataFrame
    # Only works if the HDF key is stored in 'table' format
    with pd.HDFStore(file_path, mode='r') as store:
        if store.get_storer(key).is_table:
            dfs = []
            i = 1
            for chunk in pd.read_hdf(file_path, key=key, chunksize=chunksize):
                dfs.append(chunk)
                print(f"Chunk {i} with {len(chunk)} rows read.")
                i += 1
            return pd.concat(dfs, ignore_index=True)
        else:
            # If not table format, read all at once
            return pd.read_hdf(file_path, key=key)
    store.close()

In [10]:
cleaned_path = './1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5'
upper_threshold = 450  # mm/day
lower_threshold = 0  # mm/day

In [12]:
df_data = df.copy()
df_data

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
123612355,88690050,2023-12-27,0.00
123612356,88690050,2023-12-28,0.00
123612357,88690050,2023-12-29,2.00
123612358,88690050,2023-12-30,0.00


# Data processing

In [13]:
# df_data = df_data = read_hdf_chunks(cleaned_path, key='table_data', chunksize=10_000_000)
df_data = df_data[(df_data['rain_mm']>= lower_threshold)
                  & (df_data['rain_mm']<= upper_threshold)
                  ].reset_index(drop=True)
df_data

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
123612355,88690050,2023-12-27,0.00
123612356,88690050,2023-12-28,0.00
123612357,88690050,2023-12-29,2.00
123612358,88690050,2023-12-30,0.00


In [4]:
df_info = pd.read_hdf(cleaned_path, key = 'table_info', encoding = 'utf-8')
df_info

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA
1,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA
2,00047003,PARÁ,CURUÇA,CURUÇA,-0.737500,-47.853600,ANA,HIDROWEB,PA
3,00047004,PARÁ,PRIMAVERA,PRIMAVERA,-0.929400,-47.099400,ANA,HIDROWEB,PA
4,00047005,PARÁ,MARAPANIM,MARUDA,-0.633600,-47.658300,ANA,HIDROWEB,PA
...,...,...,...,...,...,...,...,...,...
18977,S713,MATO GROSSO DO SUL,NOVA ANDRADINA,NOVA ANDRADINA | S713,-22.078611,-53.465833,INMET,INMET,MS
18978,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS
18979,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS
18980,S716,MATO GROSSO DO SUL,SANTA RITA DO PARDO,SANTA RITA DO PARDO | S716,-21.305889,-52.820375,INMET,INMET,MS


In [5]:
def max_consecutive_dry_days_numpy(rain_array):
    """Retorna o maior número de dias consecutivos com rain == 0.0.
       rain_array pode conter floats e np.nan (np.nan != 0.0)."""
    arr = np.asarray(rain_array)
    dry = (arr == 0.0)          # bool array: True onde seco
    if not dry.any():
        return 0
    padded = np.concatenate(([False], dry, [False]))
    diff = np.diff(padded.astype(int))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    lengths = ends - starts
    return int(lengths.max()) if lengths.size > 0 else 0

def compute_metrics(df, parallel=True, n_jobs=-1, tol=None):
    """
    df: DataFrame com colunas ['gauge_code', 'datetime', 'rain_mm'].
    parallel: usa joblib para processar grupos em paralelo.
    tol: se fornecido (float >=0), considera `np.isclose(val, 0.0, atol=tol)` para definir dia seco.
         Se tol is None, usa igualdade exata (val == 0.0).
    Retorna DataFrame indexed por (gauge_code, year) com colunas:
      'annual_rainfall_mm', 'active_days', 'consecutive_dry_days'
    """
    df = df.copy()
    df['year'] = df['datetime'].dt.year

    # agregados vetorizados (rápidos)
    agg = df.groupby(['gauge_code', 'year']).agg(
        annual_rainfall_mm=('rain_mm', 'sum'),
        active_days=('rain_mm', lambda x: (x >= 0).sum())
    )

    # função que calcula consecutivos considerando tol se fornecido
    if tol is None:
        def _calc_consec(arr): 
            return max_consecutive_dry_days_numpy(arr)
    else:
        def _calc_consec(arr):
            a = np.asarray(arr)
            dry = np.isclose(a, 0.0, atol=tol, rtol=0)
            if not np.any(dry):
                return 0
            padded = np.concatenate(([False], dry, [False]))
            diff = np.diff(padded.astype(int))
            starts = np.where(diff == 1)[0]
            ends = np.where(diff == -1)[0]
            lengths = ends - starts
            return int(lengths.max()) if lengths.size > 0 else 0

    # calcula consecutive_dry_days (paraleliza por grupo se solicitado)
    if parallel:
        groups = [g for _, g in df.groupby(['gauge_code', 'year'])]
        results = Parallel(n_jobs=n_jobs)(
            delayed(lambda g: _calc_consec(g['rain_mm'].to_numpy()))(g) for g in groups
        )
        index = [(g['gauge_code'].iloc[0], g['year'].iloc[0]) for g in groups]
        cons_series = pd.Series(results, index=pd.MultiIndex.from_tuples(index, names=['gauge_code', 'year']),
                                name='consecutive_dry_days')
    else:
        cons_series = df.groupby(['gauge_code', 'year'])['rain_mm'] \
                        .apply(lambda x: _calc_consec(x.to_numpy()))

    # juntar agregados
    result = agg.join(cons_series)
    # garantir tipos inteiros para consecutivos/active_days se quiser
    result['consecutive_dry_days'] = result['consecutive_dry_days'].astype(int)
    result['active_days'] = result['active_days'].astype(int)
    return result

df_preclassif = compute_metrics(df_data, parallel=True, n_jobs=-1, tol=None)
df_preclassif

annual_rainfall_mm  active_days  consecutive_dry_days
gauge_code year                                                       
00047000   1961              2186.0          365                   275
           1962               273.8          365                   153
           1963               686.2          365                   115
           1964               597.5          366                   145
00047002   1977               133.4           23                     6
...                             ...          ...                   ...
S713       2021                76.2          365                   150
S714       2021               828.0          365                    75
S715       2021              1041.8          365                    76
S716       2021               928.8          365                    68
S717       2021                 0.0          365                   365

[346029 rows x 3 columns]

In [6]:
df_preclassif = df_preclassif.reset_index(drop=False)
df_preclassif['preclassif'] = df_preclassif.apply(
    lambda row: 'LQ' if (row['annual_rainfall_mm'] < 300
                         or row['annual_rainfall_mm'] > 6000
                        #  or row['active_days'] < 305
                         or row['consecutive_dry_days'] > 200) else "", axis=1)
df_preclassif.to_hdf(cleaned_path, key='table_preclassif', mode='r+', append = False, complevel=9, complib='blosc:lz4', encoding='utf-8')
df_preclassif

,gauge_code,year,annual_rainfall_mm,active_days,consecutive_dry_days,preclassif
0,00047000,1961,2186.0,365,275,LQ
1,00047000,1962,273.8,365,153,LQ
2,00047000,1963,686.2,365,115,
3,00047000,1964,597.5,366,145,
4,00047002,1977,133.4,23,6,LQ
...,...,...,...,...,...,...
346024,S713,2021,76.2,365,150,LQ
346025,S714,2021,828.0,365,75,
346026,S715,2021,1041.8,365,76,
346027,S716,2021,928.8,365,68,


In [6]:
df_preclassif = read_hdf(cleaned_path, key='table_preclassif')
df_preclassif

,gauge_code,year,annual_rainfall_mm,active_days,consecutive_dry_days,preclassif
0,00047000,1961,2186.0,365,275,LQ
1,00047000,1962,273.8,365,153,LQ
2,00047000,1963,686.2,365,115,
3,00047000,1964,597.5,366,145,
4,00047002,1977,133.4,23,6,LQ
...,...,...,...,...,...,...
346024,S713,2021,76.2,365,150,LQ
346025,S714,2021,828.0,365,75,
346026,S715,2021,1041.8,365,76,
346027,S716,2021,928.8,365,68,


In [7]:
preclassif_counts = df_preclassif['preclassif'].value_counts()
print(preclassif_counts)

preclassif
      299777
LQ     46252
Name: count, dtype: int64


In [8]:
print(f"Percentage of LQ preclassification: {preclassif_counts['LQ'] / preclassif_counts.sum() * 100:.2f}%")

Percentage of LQ preclassification: 13.37%


# Outlier treatment

In [9]:
max_lengths = {col: df_info[col].astype(str).map(len).max() for col in df_info.columns} 
print(max_lengths)

{'gauge_code': 10, 'state': 19, 'city': 39, 'name_station': 56, 'lat': 12, 'long': 12, 'responsible': 74, 'source': 10, 'state_abbreviation': 2}


In [10]:
max_lengths = {col: df_data[col].astype(str).map(len).max() for col in df_data.columns} 
print(max_lengths)

{'gauge_code': 10, 'datetime': 10, 'rain_mm': 19}


In [17]:
df_data = read_hdf_chunks(cleaned_path, key='table_data')
df_data

ValueError: Table 'table_data' not found in file './1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5'

In [15]:
df_data = read_hdf(cleaned_path, key='table_data')
df_data

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
2908882,88690050,2023-12-27,0.00
2908883,88690050,2023-12-28,0.00
2908884,88690050,2023-12-29,2.00
2908885,88690050,2023-12-30,0.00


In [11]:
df_preclassif = pd.read_hdf(cleaned_path, key = 'table_preclassif', encoding = 'utf-8')
df_data['year'] = df_data['datetime'].dt.year
df_outlier = pd.merge(df_data, df_preclassif[['gauge_code', 'year', 'preclassif']], on = ['gauge_code', 'year'], how = 'left')
df_outlier = df_outlier[df_outlier['preclassif'] != "LQ"]
df_outlier.pop('preclassif')
df_outlier.pop('year')
df_outlier

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
123612355,88690050,2023-12-27,0.00
123612356,88690050,2023-12-28,0.00
123612357,88690050,2023-12-29,2.00
123612358,88690050,2023-12-30,0.00


In [ ]:
chunk_size = 10_000_000  # Adjust based on available memory
with pd.HDFStore(cleaned_path, mode='r+', complevel=9, complib='blosc:lz4') as store:
    for start in range(0, len(df_outlier), chunk_size):
        end = min(start + chunk_size, len(df_outlier))
        chunk = df_outlier.iloc[start:end]
        append_mode = start != 0 # Append after the first chunk
        store.append(
            'table_outlier',
            chunk,
            format='table',
            data_columns=True,
            min_itemsize={'gauge_code': 20},
            encoding='utf-8',
            append=append_mode
        )
        del chunk  # free memory
        print(f"Written rows {start} to {end}")

with h5py.File(cleaned_path, 'r') as f:
    print("Tables in cleaned_path:")
    print(list(f.keys()))

Written rows 0 to 10000000
Written rows 10000000 to 20000000
Written rows 20000000 to 30000000
Written rows 30000000 to 40000000
Written rows 40000000 to 50000000
Written rows 50000000 to 60000000
Written rows 60000000 to 70000000
Written rows 70000000 to 80000000
Written rows 80000000 to 90000000
Written rows 90000000 to 100000000
Written rows 100000000 to 107983337
Tables in cleaned_path:
['table_data', 'table_info', 'table_outlier', 'table_preclassif']


In [24]:
del df_outlier_filter_1, df_data, df_info, df_preclassif, df_outlier, df_outlier_filter_1_export

In [25]:
df_outlier = pd.read_hdf(cleaned_path, key = 'table_outlier', encoding = 'utf-8')
df_outlier

MemoryError: Unable to allocate 1.61 GiB for an array with shape (107983337,) and data type complex128

In [16]:
def mark_outlier_rain_optimized(df, threshold_rain_mm=200):
    # Garantir que a coluna 'datetime' está no formato datetime
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    # 1. Filtrar apenas registros com chuva acima do threshold
    high_rain_df = df[df['rain_mm'] > threshold_rain_mm].copy()
    
    if high_rain_df.empty:
        print("Nenhum registro com chuva acima do threshold encontrado.")
        df['outlier_status_1'] = 0
        return df
    
    # 2. Criar DataFrames com datas de ontem e amanhã para merge
    yesterday_df = high_rain_df[['gauge_code', 'datetime']].copy()
    yesterday_df['datetime'] = yesterday_df['datetime'] - pd.Timedelta(days=1)
    yesterday_df = yesterday_df.merge(
        df[['gauge_code', 'datetime', 'rain_mm']], 
        on=['gauge_code', 'datetime'], 
        how='left'
    ).rename(columns={'rain_mm': 'rain_mm_yesterday'}).fillna(0)
    
    tomorrow_df = high_rain_df[['gauge_code', 'datetime']].copy()
    tomorrow_df['datetime'] = tomorrow_df['datetime'] + pd.Timedelta(days=1)
    tomorrow_df = tomorrow_df.merge(
        df[['gauge_code', 'datetime', 'rain_mm']], 
        on=['gauge_code', 'datetime'], 
        how='left'
    ).rename(columns={'rain_mm': 'rain_mm_tomorrow'}).fillna(0)
    
    # 3. Juntar os dados adjacentes de volta ao high_rain_df
    high_rain_df = high_rain_df.merge(
        yesterday_df[['gauge_code', 'datetime', 'rain_mm_yesterday']],
        on=['gauge_code', 'datetime'],
        how='left'
    )
    
    high_rain_df = high_rain_df.merge(
        tomorrow_df[['gauge_code', 'datetime', 'rain_mm_tomorrow']],
        on=['gauge_code', 'datetime'],
        how='left'
    )
    
    # Preencher NaN com 0 caso não existam dados para os dias adjacentes
    high_rain_df[['rain_mm_yesterday', 'rain_mm_tomorrow']] = high_rain_df[
        ['rain_mm_yesterday', 'rain_mm_tomorrow']
    ].fillna(0)
    
    # 4. Calcular soma da chuva nos dias adjacentes
    high_rain_df['adjacent_days_mm'] = high_rain_df['rain_mm_yesterday'] + high_rain_df['rain_mm_tomorrow']
    
    # 5. Aplicar regra para identificar outlier
    condition = (high_rain_df['adjacent_days_mm'] < 0.025 * high_rain_df['rain_mm'])
    high_rain_df['outlier_status_1'] = np.where(condition, 1, 0)
    
    # 6. Mesclar resultados de volta ao DataFrame original
    df['outlier_status_1'] = 0
    
    # Atualizar apenas os registros que foram processados
    outlier_mask = df.index.isin(high_rain_df[high_rain_df['outlier_status_1'] == 1].index)
    df.loc[outlier_mask, 'outlier_status_1'] = 1

    df_outlier_filter_1_export = df[df['outlier_status_1'] == 1]
    df_outlier_filter_1_export.to_hdf(
        cleaned_path,
        key='table_outlier_filter_1',
        mode='r+', 
        append=False, 
        complevel=9, 
        complib='blosc:lz4', 
        format='table', 
        data_columns=True, 
        min_itemsize={'gauge_code': 20}
    )
    
    print(f"Encontrados {outlier_mask.sum()} outliers potenciais")
    print(df.head())

    return df

# Usar a função otimizada
df_outlier_filter_1 = mark_outlier_rain_optimized(df_outlier)

Encontrados 3242 outliers potenciais
   gauge_code   datetime  rain_mm  outlier_status_1
0  110018901A 2021-01-01     6.30                 1
1  110018901A 2021-01-02     1.79                 1
2  110018901A 2021-01-03     0.00                 1
3  110018901A 2021-01-04    16.15                 1
4  110018901A 2021-01-05     1.58                 1


In [17]:
df_outlier_filter_1_export = pd.read_hdf(
    cleaned_path,
    key='table_outlier_filter_1',
    encoding = 'utf-8'
)
df_outlier_filter_1_export

,gauge_code,datetime,rain_mm,outlier_status_1
0,110018901A,2021-01-01,6.30,1
1,110018901A,2021-01-02,1.79,1
2,110018901A,2021-01-03,0.00,1
3,110018901A,2021-01-04,16.15,1
4,110018901A,2021-01-05,1.58,1
...,...,...,...,...
4540,120010401A,2024-06-06,0.00,1
4541,120010401A,2024-06-07,0.00,1
4544,120010401A,2024-06-10,0.00,1
4545,120010401A,2024-06-11,0.00,1


In [20]:
df_outlier

,gauge_code,datetime,rain_mm,outlier_status_1_x,outlier_status_1_y
0,110018901A,2021-01-01,6.30,1,1.0
1,110018901A,2021-01-02,1.79,1,1.0
2,110018901A,2021-01-03,0.00,1,1.0
3,110018901A,2021-01-04,16.15,1,1.0
4,110018901A,2021-01-05,1.58,1,1.0
...,...,...,...,...,...
107983332,88690050,2023-12-27,0.00,0,NaN
107983333,88690050,2023-12-28,0.00,0,NaN
107983334,88690050,2023-12-29,2.00,0,NaN
107983335,88690050,2023-12-30,0.00,0,NaN


In [18]:
df_outlier = pd.merge(df_outlier, df_outlier_filter_1_export[['gauge_code', 'datetime', 'outlier_status_1']], on=['gauge_code', 'datetime'], how='left')
df_outlier

,gauge_code,datetime,rain_mm,outlier_status_1_x,outlier_status_1_y
0,110018901A,2021-01-01,6.30,1,1.0
1,110018901A,2021-01-02,1.79,1,1.0
2,110018901A,2021-01-03,0.00,1,1.0
3,110018901A,2021-01-04,16.15,1,1.0
4,110018901A,2021-01-05,1.58,1,1.0
...,...,...,...,...,...
107983332,88690050,2023-12-27,0.00,0,NaN
107983333,88690050,2023-12-28,0.00,0,NaN
107983334,88690050,2023-12-29,2.00,0,NaN
107983335,88690050,2023-12-30,0.00,0,NaN


In [10]:
df_outlier.groupby('outlier_status_1').size()

outlier_status_1
1.0    3242
dtype: int64

In [11]:
# df_outlier_filter_1 = df_outlier_filter_1[['gauge_code','datetime',	'rain_mm', 'outlier_status_1']]
# df_outlier_filter_1 = df_data.merge(df_outlier_filter_1_export, on = ['gauge_code', 'datetime'], how = 'left')
df_info = pd.read_hdf(cleaned_path, key = 'table_info', encoding = 'utf-8')
df_outlier_filter_1 = df_outlier[df_outlier['outlier_status_1']!=1.0] # Manter apenas registros marcados como outliers
df_outlier_filter_1 = pd.merge(df_outlier_filter_1, df_info[['gauge_code', 'lat', 'long']], on ='gauge_code', how='left')
df_outlier_filter_1.reset_index(drop=True, inplace=True)
df_outlier_filter_1

,gauge_code,datetime,rain_mm,outlier_status_1,lat,long
0,110018901A,2021-02-02,9.48,NaN,-11.683234,-61.182871
1,110018901A,2021-02-03,4.73,NaN,-11.683234,-61.182871
2,110018901A,2021-02-04,73.28,NaN,-11.683234,-61.182871
3,110018901A,2021-02-07,0.20,NaN,-11.683234,-61.182871
4,110018901A,2021-02-08,0.00,NaN,-11.683234,-61.182871
...,...,...,...,...,...,...
107980090,88690050,2023-12-27,0.00,NaN,-31.811100,-52.389200
107980091,88690050,2023-12-28,0.00,NaN,-31.811100,-52.389200
107980092,88690050,2023-12-29,2.00,NaN,-31.811100,-52.389200
107980093,88690050,2023-12-30,0.00,NaN,-31.811100,-52.389200


In [13]:
del df_outlier

In [14]:
df_outlier_filter_1.pop('outlier_status_1')

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
             ..
107980090   NaN
107980091   NaN
107980092   NaN
107980093   NaN
107980094   NaN
Name: outlier_status_1, Length: 107980095, dtype: float64

In [20]:
df_outlier_filter_1 = df_outlier_filter_1.dropna()
df_outlier_filter_1

,gauge_code,datetime,rain_mm,lat,long
0,110018901A,2021-02-02,9.48,-11.683234,-61.182871
1,110018901A,2021-02-03,4.73,-11.683234,-61.182871
2,110018901A,2021-02-04,73.28,-11.683234,-61.182871
3,110018901A,2021-02-07,0.20,-11.683234,-61.182871
4,110018901A,2021-02-08,0.00,-11.683234,-61.182871
...,...,...,...,...,...
107980090,88690050,2023-12-27,0.00,-31.811100,-52.389200
107980091,88690050,2023-12-28,0.00,-31.811100,-52.389200
107980092,88690050,2023-12-29,2.00,-31.811100,-52.389200
107980093,88690050,2023-12-30,0.00,-31.811100,-52.389200


In [21]:
def idw_interpolation(latitude, longitude, df_temp_without_gauge, kdtree, p=2):
    row = [latitude, longitude]
    distances, indices = kdtree.query(row, k=5)
    weights = 1 / (distances + 1e-6) ** p
    values = df_temp_without_gauge.iloc[indices]['rain_mm'].values
    return (np.sum(weights * values) / np.sum(weights))

# Initialize empty DataFrame for results
outlier_analysis_results = pd.DataFrame()

# Configurations
output_filename = os.path.join(neighboring_data_path, "neighboring_analysis.h5")
rainfall_threshold = 200.0  # mm

start_date = '1961-01-01'
end_date = '2024-12-31'

df_date_filter = df_outlier_filter_1.loc[(df_outlier_filter_1['datetime'] >= start_date) & (df_outlier_filter_1['datetime'] <= end_date)].sort_values('datetime', ignore_index=True, ascending=True)

df_date_filter = df_date_filter[df_date_filter['rain_mm'] > 200.0].reset_index(drop=True) # Filter for high rainfall values

# Get sorted unique dates
analysis_dates = df_date_filter['datetime'].unique().tolist() # Get unique dates for the analysis
del df_date_filter
analysis_dates.sort()

# Process each date
for current_date in analysis_dates[:]:
    # Filter data for current date
    daily_data = df_outlier_filter_1[df_outlier_filter_1['datetime'] == current_date]
    
    df_gauge_filter = daily_data[daily_data['rain_mm'] > 200.0].reset_index(drop=True) # Filter for high rainfall values
    
    gauge_codes = df_gauge_filter['gauge_code'].unique() # Get unique gauge codes for the current date
    
    date_results = []
    
    for gauge in gauge_codes:
        gauge_data = daily_data[daily_data['gauge_code'] == gauge].iloc[0]
        lat, lon = gauge_data['lat'], gauge_data['long']
        observed_rain = gauge_data['rain_mm']
        
        # Initialize result row
        result_row = {
            'gauge_code': gauge,
            'datetime': current_date,
            'lat': lat,
            'long': lon,
            'observed_rain_mm': observed_rain,
            'interpolated_rain_mm': np.nan
        }
        
        # Only interpolate for high rainfall values
        if observed_rain > rainfall_threshold:
            neighbor_data = daily_data[daily_data['gauge_code'] != gauge]
            
            if len(neighbor_data) > 0:
                kd_tree = KDTree(neighbor_data[['lat', 'long']].values)
                result_row['interpolated_rain_mm'] = idw_interpolation(lat, lon, neighbor_data, kd_tree)
        
        date_results.append(pd.DataFrame([result_row]))
    
    # Combine results for current date
    daily_results = pd.concat(date_results, ignore_index=True)
    
    # Save to HDF5 with proper configuration
    storage_mode = 'w' if current_date == analysis_dates[0] else 'a'
    append_mode = False if current_date == analysis_dates[0] else True

    # storage_mode = 'r+'
    # append_mode = True
    
    daily_results.to_hdf(
        output_filename,
        key='table_data',
        mode=storage_mode,
        format='table',
        complevel=9,
        encoding='utf-8',
        append=append_mode,
        complib='blosc:lz4',
        min_itemsize={'gauge_code': 20}  # Adjust based on your max gauge code length
    )
    
    print(f"Saved results for {current_date} to {output_filename}")

Saved results for 1961-01-02 00:00:00 to ./1 - Organized data gauge/BRAZIL\neighboring_analysis.h5
Saved results for 1961-01-04 00:00:00 to ./1 - Organized data gauge/BRAZIL\neighboring_analysis.h5
Saved results for 1961-01-05 00:00:00 to ./1 - Organized data gauge/BRAZIL\neighboring_analysis.h5
Saved results for 1961-01-11 00:00:00 to ./1 - Organized data gauge/BRAZIL\neighboring_analysis.h5
Saved results for 1961-01-12 00:00:00 to ./1 - Organized data gauge/BRAZIL\neighboring_analysis.h5
Saved results for 1961-01-14 00:00:00 to ./1 - Organized data gauge/BRAZIL\neighboring_analysis.h5
Saved results for 1961-01-23 00:00:00 to ./1 - Organized data gauge/BRAZIL\neighboring_analysis.h5
Saved results for 1961-01-25 00:00:00 to ./1 - Organized data gauge/BRAZIL\neighboring_analysis.h5
Saved results for 1961-01-26 00:00:00 to ./1 - Organized data gauge/BRAZIL\neighboring_analysis.h5
Saved results for 1961-01-27 00:00:00 to ./1 - Organized data gauge/BRAZIL\neighboring_analysis.h5
Saved resu

In [ ]:
# del df_outlier_filter_1

In [22]:
output_filename = os.path.join(neighboring_data_path, "neighboring_analysis.h5")

df_outlier_2 = pd.read_hdf(output_filename, key='table_data')
df_outlier_2

,gauge_code,datetime,lat,long,observed_rain_mm,interpolated_rain_mm
0,02244065,1961-01-02,-22.170000,-44.636900,251.20,9.687361
0,02146011,1961-01-04,-21.833300,-46.900000,218.30,21.706906
0,02241011,1961-01-05,-22.066700,-41.600000,212.20,23.561928
0,02241011,1961-01-11,-22.066700,-41.600000,360.20,11.429093
0,02143012,1961-01-12,-21.750000,-43.333300,220.00,38.436259
...,...,...,...,...,...,...
0,330300501A,2024-12-25,-21.412338,-42.196813,438.20,1.211600
0,56991380,2024-12-26,-19.971900,-41.110600,255.80,0.039461
0,290650101A,2024-12-29,-12.669000,-38.543000,264.72,0.000000
1,56991380,2024-12-29,-19.971900,-41.110600,408.40,0.655309


In [23]:
df_outlier_filter_2_export = df_outlier_2[df_outlier_2['interpolated_rain_mm'] >= 0.0].reset_index(drop = True)
df_outlier_filter_2_export = df_outlier_filter_2_export[df_outlier_filter_2_export['interpolated_rain_mm'] >= 0.35 * df_outlier_filter_2_export['observed_rain_mm']]
df_outlier_filter_2_export['outlier_status_2'] = 1
df_outlier_filter_2_export

,gauge_code,datetime,lat,long,observed_rain_mm,interpolated_rain_mm,outlier_status_2
7,02346034,1961-01-23,-23.4167,-46.5667,200.30,128.216270,1
10,02346162,1961-01-26,-23.7000,-46.0667,231.00,182.781510,1
11,02346223,1961-01-26,-23.7000,-46.0167,272.60,151.228139,1
12,02345145,1961-01-27,-23.5997,-45.9089,201.40,71.139330,1
17,02447038,1961-02-16,-24.8000,-47.9667,252.60,113.109432,1
...,...,...,...,...,...,...,...
4432,65365801,2024-05-27,-26.1653,-51.2281,315.60,190.400175,1
4435,65803001,2024-05-27,-25.7883,-52.1147,353.20,126.110557,1
4496,62780700,2024-11-04,-21.8194,-48.8281,218.20,120.550931,1
4526,65824880,2024-12-09,-25.6406,-51.8603,209.60,78.153355,1


In [ ]:
# del df_outlier_2

In [24]:
df_outlier_filter_2_export.to_hdf(
        os.path.join(neighboring_data_path, "neighboring_analysis_filter_2.h5"),
        key='table_data',
        mode='w',
        format='table',
        complevel=9,
        encoding='utf-8',
        append=False,
        min_itemsize={'gauge_code': 20}  # Adjust based on your max gauge code length
    )
df_outlier_filter_2_export

,gauge_code,datetime,lat,long,observed_rain_mm,interpolated_rain_mm,outlier_status_2
7,02346034,1961-01-23,-23.4167,-46.5667,200.30,128.216270,1
10,02346162,1961-01-26,-23.7000,-46.0667,231.00,182.781510,1
11,02346223,1961-01-26,-23.7000,-46.0167,272.60,151.228139,1
12,02345145,1961-01-27,-23.5997,-45.9089,201.40,71.139330,1
17,02447038,1961-02-16,-24.8000,-47.9667,252.60,113.109432,1
...,...,...,...,...,...,...,...
4432,65365801,2024-05-27,-26.1653,-51.2281,315.60,190.400175,1
4435,65803001,2024-05-27,-25.7883,-52.1147,353.20,126.110557,1
4496,62780700,2024-11-04,-21.8194,-48.8281,218.20,120.550931,1
4526,65824880,2024-12-09,-25.6406,-51.8603,209.60,78.153355,1


In [25]:
df_outlier_filter_2_export = pd.read_hdf(os.path.join(neighboring_data_path, "neighboring_analysis_filter_2.h5"), key = 'table_data', encoding = 'utf-8')
df_outlier_filter_2_export

,gauge_code,datetime,lat,long,observed_rain_mm,interpolated_rain_mm,outlier_status_2
7,02346034,1961-01-23,-23.4167,-46.5667,200.30,128.216270,1
10,02346162,1961-01-26,-23.7000,-46.0667,231.00,182.781510,1
11,02346223,1961-01-26,-23.7000,-46.0167,272.60,151.228139,1
12,02345145,1961-01-27,-23.5997,-45.9089,201.40,71.139330,1
17,02447038,1961-02-16,-24.8000,-47.9667,252.60,113.109432,1
...,...,...,...,...,...,...,...
4432,65365801,2024-05-27,-26.1653,-51.2281,315.60,190.400175,1
4435,65803001,2024-05-27,-25.7883,-52.1147,353.20,126.110557,1
4496,62780700,2024-11-04,-21.8194,-48.8281,218.20,120.550931,1
4526,65824880,2024-12-09,-25.6406,-51.8603,209.60,78.153355,1


# Data Deletion

In [28]:
df_filter_1 = pd.read_hdf(
    r"./1 - Organized data gauge/BRAZIL/outlier_processing.h5",
    key='table_outlier_filter_1',
    encoding = 'utf-8'
)
df_filter_1

,gauge_code,datetime,rain_mm,outlier_status_1
0,110018901A,2021-01-01,6.30,1
1,110018901A,2021-01-02,1.79,1
2,110018901A,2021-01-03,0.00,1
3,110018901A,2021-01-04,16.15,1
4,110018901A,2021-01-05,1.58,1
...,...,...,...,...
4540,120010401A,2024-06-06,0.00,1
4541,120010401A,2024-06-07,0.00,1
4544,120010401A,2024-06-10,0.00,1
4545,120010401A,2024-06-11,0.00,1


In [29]:

df_filter_2 = pd.read_hdf(os.path.join(neighboring_data_path, "neighboring_analysis_filter_2.h5"), key='table_data')
df_filter_2['outlier_status_2'] = 1
df_filter_2 = df_filter_2[['gauge_code', 'datetime', 'outlier_status_2']]
df_filter_2

,gauge_code,datetime,outlier_status_2
7,02346034,1961-01-23,1
10,02346162,1961-01-26,1
11,02346223,1961-01-26,1
12,02345145,1961-01-27,1
17,02447038,1961-02-16,1
...,...,...,...
4432,65365801,2024-05-27,1
4435,65803001,2024-05-27,1
4496,62780700,2024-11-04,1
4526,65824880,2024-12-09,1


In [30]:
df_data = pd.read_hdf(cleaned_path, key = 'table_data', encoding = 'utf-8')
df_data

,gauge_code,datetime,rain_mm
0,110018901A,2021-01-01,6.30
1,110018901A,2021-01-02,1.79
2,110018901A,2021-01-03,0.00
3,110018901A,2021-01-04,16.15
4,110018901A,2021-01-05,1.58
...,...,...,...
2908882,88690050,2023-12-27,0.00
2908883,88690050,2023-12-28,0.00
2908884,88690050,2023-12-29,2.00
2908885,88690050,2023-12-30,0.00


In [ ]:
df_data_filtered = pd.merge(df_data, df_filter_1, on = ['gauge_code', 'datetime'], how = 'left').merge(df_filter_2, on = ['gauge_code', 'datetime'], how = 'left')
df_data_filtered = df_data_filtered[(df_data_filtered['outlier_status_1'] != 1) & (df_data_filtered['outlier_status_2'] != 1)]
# df_data_filtered = df_data_filtered[['gauge_code', 'datetime', 'rain_mm']].copy(deep = True)
df_data_filtered

KeyError: "['rain_mm'] not in index"

In [33]:
df_data_filtered = df_data_filtered.rename(columns={'rain_mm_x': 'rain_mm'})
df_data_filtered = df_data_filtered[['gauge_code', 'datetime', 'rain_mm']]
df_data_filtered

,gauge_code,datetime,rain_mm
32,110018901A,2021-02-02,9.48
33,110018901A,2021-02-03,4.73
34,110018901A,2021-02-04,73.28
37,110018901A,2021-02-07,0.20
38,110018901A,2021-02-08,0.00
...,...,...,...
123612355,88690050,2023-12-27,0.00
123612356,88690050,2023-12-28,0.00
123612357,88690050,2023-12-29,2.00
123612358,88690050,2023-12-30,0.00


In [35]:
key = 'table_data'
chunk_size = 10_000_000  # Adjust based on available memory

with pd.HDFStore(r"./1 - Organized data gauge/BRAZIL/BRAZIL_DAILY_1961_2024_CLEANED_OUTLIER.h5", mode='w', complevel=9, complib='blosc:lz4') as store:
    for start in range(0, len(df_data_filtered), chunk_size):
        i = start // chunk_size
        end = start + chunk_size
        chunk = df_data_filtered.iloc[start:end]
        if i == 0:
            append_mode = False
        else:  
            append_mode = True
        # Append the chunk to the HDF5 file
        store.append(key, chunk, format='table', data_columns=True, encoding='utf-8', min_itemsize={'gauge_code': 20}, append=append_mode)

        print(f"Chunk {i + 1} of {len(df_data_filtered) // chunk_size + 1} completed.")

Chunk 1 of 13 completed.
Chunk 2 of 13 completed.
Chunk 3 of 13 completed.
Chunk 4 of 13 completed.
Chunk 5 of 13 completed.
Chunk 6 of 13 completed.
Chunk 7 of 13 completed.
Chunk 8 of 13 completed.
Chunk 9 of 13 completed.
Chunk 10 of 13 completed.
Chunk 11 of 13 completed.
Chunk 12 of 13 completed.
Chunk 13 of 13 completed.


In [36]:
df_data_filtered['gauge_code'].nunique()

18503